# 00 · Colab setup and environment check

Mounts Drive, clones the repo and installs the pinned requirements. Then it checks the GPU and logs the exact library versions to Drive. **Runtime → Change runtime type → GPU** first.

In [ ]:
# --- Setup: mount Drive, clone or update the repo, install pinned deps. Safe to rerun. ---
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Pranav1011/ie7615-generative-project.git"
BRANCH = "main"
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

def _git(*args, url=None):
    """Run git. For a private repo, add a GITHUB_TOKEN Colab secret (key icon in the sidebar)."""
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    auth_url = url.replace("https://", f"https://x-access-token:{token}@") if (url and token) else url
    cmd = ["git", *[auth_url if a == "<URL>" else a for a in args]]
    res = subprocess.run(cmd, capture_output=True, text=True)
    out = (res.stdout + res.stderr).replace(token, "***") if token else res.stdout + res.stderr
    if res.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/ie7615-generative-project")
    if (REPO_DIR / ".git").exists():
        print(_git("-C", str(REPO_DIR), "pull", "--ff-only", "<URL>", BRANCH, url=REPO_URL))
    else:
        _git("clone", "-q", "-b", BRANCH, "<URL>", str(REPO_DIR), url=REPO_URL)
        _git("-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL)  # never persist a token
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("repo:", REPO_DIR, "| commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
from src.utils import load_config, get_paths, set_seed
cfg = load_config()
paths = get_paths(cfg)
set_seed(cfg["seed"])
print("artifact root:", paths["root"])

In [ ]:
import torch
from src.utils import environment_info, save_json

env = environment_info()
for k, v in env.items():
    print(f"{k}: {v}")
if not torch.cuda.is_available():
    print("\nNo GPU found. Switch the runtime to GPU before running notebooks 02 and 03.")
else:
    free, total = torch.cuda.mem_get_info()
    print(f"\nGPU memory: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")

In [ ]:
# Record the environment of this session (versions, GPU) next to the results.
print("saved:", save_json({"notebook": "00_colab_setup"}, paths["outputs"] / "env_00_colab_setup.json", add_env=True))

In [ ]:
# Optional: pre-download the model weights into the HF cache (~3 GB total) to speed up 02 and 03.
from huggingface_hub import snapshot_download
for repo in (cfg["models"]["clip"], cfg["models"]["bert"]):
    snapshot_download(repo, allow_patterns=["*.json", "*.txt", "*.safetensors"])
snapshot_download(cfg["models"]["diffusion"], allow_patterns=["*.json", "*.txt", "*fp16.safetensors"])
print("model weights cached")